# W&B Sweep Report

Queries wandb sweep results and builds a summary table + Report. Self-contained: does not depend on `criteo_functions.py` or the other notebooks.

In [ ]:
import wandb
import polars as pl
import polars.selectors as cs
import pandas as pd
import wandb_workspaces.reports.v2 as wr
import altair as alt
pl.Config.set_tbl_rows(100)

## Download Functions

In [ ]:
def get_project_sweeps(
    entity: str = "seanv507",
) -> dict[str, list[dict]]:
    """
    Return every accessible project under `entity` and its sweep IDs.

    Projects without sweeps are included with an empty list.
    """
    api = wandb.Api(timeout=60)

    return {
        project.name: [
            {
                "sweep_id": sweep.id,
                "sweep_config": sweep.config,
                "sweep_runs": [{
                    "run_id": run.id,
                    "run_summary_metrics": run.summary_metrics
                } for run in sweep.runs]
            }
            for sweep in project.sweeps(per_page=100)
        ]
        for project in api.projects(entity=entity, per_page=200)
    }


def get_project_runs(
    entity: str = "seanv507",
    project_name: str | None = None,
    filters: dict | None =None,
) -> dict[str, list[dict]]:
    """
    Return every accessible project under `entity` and its runs, whether or
    not they belong to a sweep.

    `project_name` is an optional regex (matched with `re.search`) used to
    select only projects whose name matches. Projects without runs are
    included with an empty list. Runs outside a sweep have `sweep_id=None`.
    """
    api = wandb.Api(timeout=60)
    pattern = re.compile(project_name) if project_name else None

    return {
        project.name: [
            {
                "run_id": run.id,
                "sweep_id": run.sweep.id if run.sweep else None,
                "state": run.state,
                "name": run.name,
                "created_at": run.created_at,
                "job_type": run.job_type,
                "tags": run.tags,
                "notes": run.notes,
                "group": run.group,
                "config": dict(run.config),
                "summary_metrics": {
                    k: v
                    for k, v in run.summary_metrics.items()
                    if isinstance(v, (int, float, str, bool)) or v is None
                },
            }
            for run in api.runs(f"{entity}/{project.name}", per_page=200, filters=filters)
        ]
        for project in api.projects(entity=entity, per_page=200)
        if pattern is None or pattern.search(project.name)
    }



## Download Action

In [ ]:
# dl_all = get_project_runs()

In [ ]:
dl = get_project_runs(project_name="criteo_boost_gpu_categorical",)

## Processing Functions

In [ ]:
import re

def make_sweeps_df(project_sweeps: dict[str, list[dict]]):
    project_sweeps_df = (
        pl.DataFrame({'project':project_sweeps.keys(), 'sweeps': project_sweeps.values()},strict=False)
        .explode('sweeps').unnest('sweeps').explode('sweep_runs').unnest('sweep_runs')
        .with_columns(pl.col('run_id'), 
                    (pl.col('sweep_config').struct['parameters']),
                    training_logloss=pl.col('run_summary_metrics').struct['validation_0-logloss.min'],
                    validation_logloss=pl.col('run_summary_metrics').struct['validation_1-logloss.min']
                    )
        .unnest('parameters')
        .sort('validation_logloss')
        
    )
    return project_sweeps_df


def flatten(mapping, prefix=""):
    # recursively flatten
    result = {}
    for key, value in mapping.items():
        name = f"{prefix}{key}"
        if isinstance(value, dict):
            result.update(flatten(value, f"{name}."))
        else:
            result[name] = value
    return result

def unnest_full(df, separator='.'):
    while True:
        df_flatten = df.unnest(separator=separator)
        if df_flatten.columns == df.columns:
            break
        df = df_flatten
    return df_flatten

def make_runs_df(project_runs: dict[str, list[dict]]):
    run_dfs = [pl.DataFrame(runs, strict=False).insert_column(0, pl.lit(project).alias("project")) for project, runs in project_runs.items()]
    runs = (
        pl.concat(run_dfs, how='vertical_relaxed')
        #.explode("runs").unnest("runs")
        .drop_nulls(subset="run_id")  # projects with no runs
        # .unnest("summary_metrics", separator=":")
    )
    runs = unnest_full(runs)
    return runs

def convertible(df: pl.DataFrame, dtype) -> list[str]:
    str_cols = df.select(pl.col(pl.String)).columns
    res = df.select(
        (pl.col(c).cast(dtype, strict=False).null_count() == pl.col(c).null_count()).alias(c)
        for c in str_cols
    )
    return [k for k,v in res.row(0, named=True).items() if v]

def convert_int_double_strs(df):
    int_convertible = convertible(df, pl.Int64)
    float_convertible = [col for col in convertible(df, pl.Float64) if col not in int_convertible]
    df = df.with_columns(pl.col(int_convertible).cast(pl.Int64), pl.col(float_convertible).cast(pl.Float64))
    return df

def identify_variation(runs_df: pl.DataFrame):
    runs_unique = runs_df.select(pl.all().value_counts(sort=True).implode())
    return runs_unique

def one_at_a_time(grid, params):
    """One row per 1-D slice: all params but `varying` fixed, `varying` takes >1 value.    
    """
    #     Columns: the varying column is set to null (it's the free axis), and the other parameter columns hold the values held fixed.
    # values dtype: it's cast to Float64 so the per-parameter frames can be stacked, because reg_lambda is int and min_split_loss is float.
    # Slices can share a run. For example, the run at (25600, 400, 5.0, 1.0) belongs to the cat_reg_lambda, min_split_loss and min_child_weight slices.
    #  Repeated settings: settings with len 2, like (0, 10, …) and (12800, 100, …), count once in values because of unique(). Their runs show up when you join back, as below.

    metric = "validation log loss" #"summary_metrics.validation_1-logloss.min"
    return pl.concat(
        [
            grid.group_by(p for p in params if p != v)
            .agg(pl.col(v).unique().sort().cast(pl.Float64).alias("values"),
                 pl.col("run_id"), pl.col(metric))
            .filter(pl.col("values").list.len() > 1)
            .with_columns(pl.lit(v).alias("varying"), pl.lit(None).alias(v))
            .select("varying", *params, "values","run_id", metric)
            for v in params
        ],
        how="vertical_relaxed",
    ).sort("varying", pl.col("values").list.len(), descending=[False, True])

def replace_strings(name: str):
    replacements = {
        "config.": "",
        "learner.gradient_booster.tree_train_param.": "",
        "summary_metrics.validation_1-logloss.min": "validation log loss"
    }

    for pattern, repl in replacements.items():
        name = re.sub(pattern, repl, name)
    return name



In [ ]:
dl_criteo_boost = {key: runs for key, runs in dl_all.items() if key.startswith("criteo_boost")}

In [ ]:
dl_criteo_boost.keys()

In [ ]:
criteo_boost_df = make_runs_df(dl_criteo_boost)

In [ ]:
criteo_boost_df = convert_int_double_strs(criteo_boost_df)

In [ ]:
criteo_boost_df=pl.read_parquet("criteo_boost.parquet")

In [ ]:
criteo_boost_df.write_parquet("criteo_boost.parquet")

In [ ]:
plot_df = (
    criteo_boost_df
    .filter(pl.col("config.sample_every")==1)
    .group_by("project")
    .agg(pl.col('summary_metrics.validation_1-logloss.min').min())
    .sort('summary_metrics.validation_1-logloss.min')
)
print(plot_df)
(
    alt.Chart(plot_df)
    .mark_bar(clip=True)
    .encode(x=alt.X("project:N",sort=None), y=alt.Y(r"summary_metrics\.validation_1-logloss\.min", scale=alt.Scale(domain=[.43, .48])))
)

In [ ]:
from IPython.display import display, Markdown

with pl.Config() as cfg:
    cfg.set_tbl_formatting('ASCII_MARKDOWN')
    display(Markdown(repr(plot_df)))

In [ ]:
criteo_boost_gpu_categorical = criteo_boost_df.filter(
    (pl.col("project")=="criteo_boost_gpu_categorical") & 
    (pl.col("config.learner.gradient_booster.tree_train_param.max_cat_to_onehot")==10) &
    (pl.col("state")=="finished") & 
    (pl.col("config.sample_every")==1)
)

OHE is now enabled by default with tree_train_param.max_cat_to_onehot set to MAX_INT
so we filter that out

We also have sample every = 100 (around 30 runs)

In [ ]:

agg_cols = ["config.cat_cutoff", "config.learner.gradient_booster.tree_train_param.cat_order", ]
plot_df =(
    criteo_boost_gpu_categorical
    .group_by(agg_cols)
    .agg(pl.len(), pl.col('summary_metrics.validation_1-logloss.min').min())
    .sort(["config.cat_cutoff", 'summary_metrics.validation_1-logloss.min'])
)

In [ ]:
with pl.Config() as cfg:
    cfg.set_tbl_formatting('ASCII_MARKDOWN')
    display(Markdown(repr(plot_df)))
    

### Breakdown


In [ ]:
cat_cutoff_dfs = criteo_boost_gpu_categorical.partition_by(
    ["config.cat_cutoff", 
     'config.learner.gradient_booster.tree_train_param.cat_order',], as_dict=True)
cat_cutoff_dfs.keys()

In [ ]:

for key, cat_cutoff_df in (((100, 'tree'), cat_cutoff_dfs[(100, 'tree')]),):#cat_cutoff_dfs.items():
    key_str = f"cat_cutoff: {key[0]} cat_order: {key[1]}"
    cat_cutoff_df = cat_cutoff_dfs[key]
    duplicate_cols = ["config.gamma","config.learner.gradient_booster.tree_train_param.gamma", "config.max_depth", "config.reg_lambda", 
                    "config.learning_rate", "config.min_child_weight",
                    "config.subsample"]
    #'config.learner.gradient_booster.tree_train_param.gamma' = min_split_loss
    config_values = (
        cat_cutoff_df
        .select(
            cs.starts_with("config.")
            .value_counts(sort=True)
            .implode()
        ).drop(duplicate_cols)
        .rename(replace_strings)
    )
    not_unique = config_values.select(pl.all().list.len()>1).row(0,named=True)
    not_unique = [k for k, v  in not_unique.items() if v ]
    baseline = (
        config_values
        .select(pl.col(not_unique).list[0].struct[0])
        .rename(replace_strings) # in struct
        .row(0, named=True)
    )
    grid = (
        cat_cutoff_df
        .drop(duplicate_cols)
        .rename(replace_strings)
        .group_by(baseline.keys()).agg("run_id",pl.col("validation log loss").mean(),pl.len()).sort(baseline.keys())
    )
    slices = one_at_a_time(grid, list(baseline.keys()))
    slices  = slices.with_columns(
        pl.concat_str(
            [pl.format(f"{f}: {{}}", pl.col(f)) for f in baseline.keys()],
            separator=", ",ignore_nulls=True
        ).alias("fixed params")
    )
    plot_dfs = {}

    for varying in slices["varying"].value_counts(sort=True)['varying']:

        plot_df = (
            slices.filter(pl.col("varying")==varying)
            .drop(varying)
            .rename({"values":varying})
            .explode(varying, "run_id", "validation log loss")
        )
        plot_dfs[varying] = plot_df
    ranges = {v: (d["validation log loss"].min(), d["validation log loss"].max()) for v, d in plot_dfs.items()}
    span = 1.1 * max(hi - lo for lo, hi in ranges.values())   # 10% padding
    best = min(lo for lo, _ in ranges.values())
    charts = {}
    for varying, plot_df in plot_dfs.items():
        lo, hi = ranges[varying]
        mid = (lo + hi) / 2
        base = alt.Chart(plot_df, title=varying).encode(
            x=f"{varying}:Q",
            y=alt.Y(
                "validation log loss:Q",
                #scale=alt.Scale(zero=False, domain=[.44,.48])
                scale=alt.Scale(domain=[mid - span / 2, mid + span / 2], nice=False, zero=False),
            ),
            color=alt.Color(
                "fixed params",
                legend=alt.Legend(labelLimit=0, labelExpr="split(datum.label, ', ')", symbolOffset=0, rowPadding=8),
            ),
        ).properties(width=350, height=300)
        best_rule = alt.Chart().mark_rule(strokeDash=[4, 4], color="gray").encode(y=alt.datum(best))
        charts[varying] = base.mark_line(clip=True) + base.mark_point(clip=True) + best_rule

    panel = (
        alt.concat(*charts.values(), columns=2, title = key_str).resolve_scale(color="independent", y="independent")
    )
    panel.save(f"figures/{key_str}.svg")

In [ ]:
duplicates

In [ ]:
criteo_boost_df.filter(pl.col("run_id")=="xnxw8gao").drop(duplicate_cols).select(cs.starts_with("config")).rename(replace_strings)

In [ ]:
grid

In [ ]:
!uv add vl-convert-python

In [ ]:
span

In [ ]:
config_values

In [ ]:
panel

In [ ]:
panel.save("figures/cat_cutoff 500 cat_order=node.html")

In [ ]:
charts.keys()

To get the actual runs for one slice, for example to plot logloss against the varying parameter, join the fixed values back to the run data:



```{python}
row = slices.row(0, named=True)
v = row["varying"]
fixed = {p: row[p] for p in baseline if p != v}
runs = cat_cutoff_dfs[(500, "node")].filter(**fixed).sort(v)
```
filter(**kwargs) works as an equality filter here, even with the dotted column names, because they're passed as dict keys.

In [ ]:
[c for c in  cat_cutoff_dfs[(500, "node")].columns if "gamma" in c]

In [ ]:
row = slices.row(0, named=True)
v = row["varying"]
fixed = {p: row[p] for p in baseline1 if p != v}
duplicate_cols = ["config.gamma", "config.max_depth", "config.reg_lambda", 
                  "config.learning_rate", "config.min_child_weight",
                  "config.subsample"]
runs = cat_cutoff_dfs[(500, "node")].drop(duplicate_cols).rename(replace_strings).filter(**fixed).sort(v)
fixed_title = ", ".join([f"{p}: {v}" for p, v in fixed.items()])

In [ ]:
fixed_title

In [ ]:
runs

In [ ]:
plot_df.filter(pl.col("varying")=="cat_reg_lambda")

In [ ]:
plot_df = slices[1].explode("values", "run_id", "summary_metrics.validation_1-logloss.min")
varying = plot_df["varying"].item(0)
plot_df = plot_df.drop(varying).rename({"values": varying})
(
    alt.Chart(plot_df)
    .mark_line(clip=True)
    .encode(x=varying, y=alt.Y(r"summary_metrics\.validation_1-logloss\.min", scale=alt.Scale(domain=[.43, .50]))
                #.encode(x=alt.X("project:N",sort=None), y=alt.Y(r"summary_metrics\.validation_1-logloss\.min", scale=alt.Scale(domain=[.43, .48])))
            )
)

In [ ]:
grid

In [ ]:
zz = identify_variation(cat_cutoff_dfs[(500,'node')])
zz_multiple = zz.select(col.name for col in zz.select(pl.all().list.len()) if (col.item()>1 and col.name.startswith('config.')))
zz_multiple.columns

In [ ]:
zz_multiple.select()

In [ ]:
zz_multiple

In [ ]:
zz_multiple.row(0, named=True)

In [ ]:
zz_multiple.select(
    pl.all().list.eval(pl.element().struct[0]).name.suffix(".value"),
    pl.all().list.eval(pl.element().struct[1])
)#.transpose(include_header=True)

In [ ]:
zz_multiple_config['config.sample_every']

In [ ]:
zz_multiple_config = zz_multiple.select(cs.starts_with("config."))
zz_multiple_config.columns

In [ ]:
select_columns = [
    # 'run_id',
    # 'state',
    # 'name',
    # 'created_at',
    # 'tags',
    # 'notes',
    # 'group',
    # 'config.learner.gradient_booster.tree_train_param.eta',
    'config.learner.gradient_booster.tree_train_param.gamma',
    # 'config.learner.gradient_booster.tree_train_param.lambda',
    'config.learner.gradient_booster.tree_train_param.cat_order',
    'config.learner.gradient_booster.tree_train_param.reg_lambda',
    'config.learner.gradient_booster.tree_train_param.learning_rate',
    'config.learner.gradient_booster.tree_train_param.cat_reg_lambda',
    'config.learner.gradient_booster.tree_train_param.min_split_loss',
    'config.learner.gradient_booster.tree_train_param.min_child_weight',
    'config.learner.gradient_booster.tree_train_param.max_cat_threshold',
    'config.learner.gradient_booster.tree_train_param.max_cat_to_onehot',
    # 'config.data_dir',
    'config.sample_every',
    # 'summary_metrics._runtime',
    'summary_metrics._step',
    # 'summary_metrics._timestamp',
    # 'summary_metrics._wandb.runtime',
    'summary_metrics.best_iteration',
    'summary_metrics.best_score',
    'summary_metrics.epoch',
]

In [ ]:
zz_multiple.columns

In [ ]:
zz_multiple

In [ ]:
select_columns = ['project',
 'run_id',
 'sweep_id',
 'state',
 'created_at',
 'job_type',

 'group',
 'config.learner.metrics',
 'config.learner.objective.name',
 'config.learner.objective.reg_loss_param.scale_pos_weight',
 'config.learner.generic_param.seed',
 'config.learner.generic_param.device',
 'config.learner.generic_param.n_jobs',
 'config.learner.generic_param.nthread',
 'config.learner.generic_param.rng_state',
 'config.learner.generic_param.random_state',
 'config.learner.generic_param.seed_per_iteration',
 'config.learner.generic_param.validate_parameters',
 'config.learner.generic_param.fail_on_invalid_gpu_id',
 'config.learner.gradient_booster.name',
 'config.learner.gradient_booster.updater',
 'config.learner.gradient_booster.dart_train_param.one_drop',
 'config.learner.gradient_booster.dart_train_param.rate_drop',
 'config.learner.gradient_booster.dart_train_param.skip_drop',
 'config.learner.gradient_booster.dart_train_param.sample_type',
 'config.learner.gradient_booster.dart_train_param.normalize_type',
 #'config.learner.gradient_booster.tree_train_param.eta',
 'config.learner.gradient_booster.tree_train_param.alpha',
 'config.learner.gradient_booster.tree_train_param.gamma',
 #'config.learner.gradient_booster.tree_train_param.lambda',
 #'config.learner.gradient_booster.tree_train_param.max_bin',
 'config.learner.gradient_booster.tree_train_param.cat_order',
 'config.learner.gradient_booster.tree_train_param.max_depth',
 'config.learner.gradient_booster.tree_train_param.reg_alpha',
 'config.learner.gradient_booster.tree_train_param.subsample',
 'config.learner.gradient_booster.tree_train_param.max_leaves',
 'config.learner.gradient_booster.tree_train_param.reg_lambda',
 'config.learner.gradient_booster.tree_train_param.grow_policy',
 'config.learner.gradient_booster.tree_train_param.refresh_leaf',
 'config.learner.gradient_booster.tree_train_param.learning_rate',
 'config.learner.gradient_booster.tree_train_param.cat_reg_lambda',
 'config.learner.gradient_booster.tree_train_param.max_delta_step',
 'config.learner.gradient_booster.tree_train_param.min_split_loss',
 'config.learner.gradient_booster.tree_train_param.sampling_method',
 'config.learner.gradient_booster.tree_train_param.colsample_bynode',
 'config.learner.gradient_booster.tree_train_param.colsample_bytree',
 'config.learner.gradient_booster.tree_train_param.min_child_weight',
 'config.learner.gradient_booster.tree_train_param.sparse_threshold',
 'config.learner.gradient_booster.tree_train_param.colsample_bylevel',
 'config.learner.gradient_booster.tree_train_param.max_cat_threshold',
 'config.learner.gradient_booster.tree_train_param.max_cat_to_onehot',
 'config.learner.gradient_booster.tree_train_param.monotone_constraints',
 'config.learner.gradient_booster.tree_train_param.interaction_constraints',
 'config.learner.gradient_booster.specified_updater',
 'config.learner.gradient_booster.gbtree_model_param.num_trees',
 'config.learner.gradient_booster.gbtree_model_param.num_parallel_tree',
 'config.learner.gradient_booster.gbtree_train_param.updater',
 'config.learner.gradient_booster.gbtree_train_param.tree_method',
 'config.learner.gradient_booster.gbtree_train_param.updater_seq',
 'config.learner.gradient_booster.gbtree_train_param.process_type',
 'config.learner.learner_model_param.num_class',
 'config.learner.learner_model_param.base_score',
 'config.learner.learner_model_param.num_target',
 'config.learner.learner_model_param.num_feature',
 'config.learner.learner_model_param.boost_from_average',
 'config.learner.learner_train_param.booster',
 'config.learner.learner_train_param.num_class',
 'config.learner.learner_train_param.objective',
 'config.learner.learner_train_param.base_score',
 'config.learner.learner_train_param.num_target',
 'config.learner.learner_train_param.multi_strategy',
 'config.learner.learner_train_param.boost_from_average',
 'config.learner.learner_train_param.disable_default_eval_metric',
 'config.version',
 'config.data_dir',
 'config.cat_cutoff',
 'config.sample_every',
 'config.cat_pseudo_count',
 'summary_metrics._runtime',
 'summary_metrics._step',
 'summary_metrics._timestamp',
 'summary_metrics._wandb.runtime',
 'summary_metrics.best_iteration',
 'summary_metrics.best_score',
 'summary_metrics.epoch',
 'summary_metrics.validation_0-logloss.min',
 'summary_metrics.validation_1-logloss.min'
  ]

In [ ]:
zz.columns

In [ ]:
n_vars = uniques.select(pl.all().list.len()).transpose(include_header=True, column_names=["distinct_count"])
n_vars

In [ ]:
n_vars

In [ ]:
uniques = identify_variation(dl_df)
uniques

In [ ]:
n_unique = dl_df.select(pl.all().n_unique())

In [ ]:
df = pl.DataFrame({"a": [0, 0,0], "b": [0, 0, 1], "c": [0, 1, 2]})
df.select([pl.col(c).value_counts().implode() for c in ["a", "b"]])

In [ ]:
n_unique

In [ ]:
dl_df.head()

In [ ]:
dl[0]

dict_keys([
    'config.cat_cutoff', 
    'config.data_dir', 
    'config.learner.generic_param.device', 
    'config.learner.gradient_booster.gbtree_train_param.updater', 
    'config.learner.gradient_booster.gbtree_train_param.updater_seq', 
    'config.learner.gradient_booster.tree_train_param.cat_order', 
    'config.learner.gradient_booster.tree_train_param.cat_reg_lambda', 
    'config.learner.gradient_booster.tree_train_param.eta', 
    'config.learner.gradient_booster.tree_train_param.lambda', 
    'config.learner.gradient_booster.tree_train_param.learning_rate', 
    'config.learner.gradient_booster.tree_train_param.max_cat_threshold', 
    'config.learner.gradient_booster.tree_train_param.max_cat_to_onehot', 
    'config.learner.gradient_booster.tree_train_param.reg_lambda', 
    'config.learner.gradient_booster.updater', 
    'config.sample_every', 
    'id', 
    'name', 
    'summary.Feature Importance_table._latest_artifact_path', 'summary.Feature Importance_table._type', 'summary.Feature Importance_table.artifact_path', 'summary.Feature Importance_table.log_mode', 'summary.Feature Importance_table.ncols', 'summary.Feature Importance_table.nrows', 'summary.Feature Importance_table.path', 'summary.Feature Importance_table.sha256', 'summary.Feature Importance_table.size', 'summary._step', 'summary.best_iteration', 'summary.best_score', 'summary.epoch', 'summary.validation_0-logloss.min', 'summary.validation_1-logloss.min', 'tags'])

In [ ]:
import json
import yaml
const_str = yaml.safe_dump(cc["constant_columns"])

In [ ]:
with open("constants.yaml", "w") as f:
    f.write(const_str)

In [ ]:
print(const_str)

In [ ]:
cc["varying_columns"].keys()

```
run = api.run('seanv507/criteo_boost_sweep_04/auwure38')
```

In [ ]:
ps = get_project_sweeps("seanv507")

In [ ]:
prs = get_project_runs("seanv507", "criteo_boost*")


In [ ]:
project_runs = make_runs_df(prs)

In [ ]:
project_runs.write_excel("project_runs.xlsx")

In [ ]:
print(project_runs.shape)
project_runs.head()

In [ ]:
project_sweeps = make_sweeps_df(ps)

In [ ]:
project_sweeps["sweep_id"].unique()

In [ ]:

project_sweeps.null_count()

In [ ]:
project_sweeps.head()

In [ ]:
project_sweeps.with_columns(pl.col('sweep_config').struct['parameters']).unnest('parameters')

In [ ]:
a1 = project_sweeps.explode('sweeps').unnest('sweeps').explode('sweep_runs').unnest('sweep_runs')


> **⚠️ Syntax error below** — `a2 = a1.` is an incomplete statement (trailing `.` with nothing after it). It looks like it was meant to continue the explode/unnest chain from `a1` above. Complete the expression or remove the cell.

In [ ]:
a2 = a1.
a2.head()

In [ ]:
a2.shape

In [ ]:
a1.shape

In [ ]:
a1.item(0,3)

In [ ]:
project_sweeps_e = project_sweeps.explode("sweep_ids")
project_sweeps_e.head()

In [ ]:
project_sweeps_e.group_by(pl.col("project").str.slice(0,10)).len().sort("len",descending=True)

In [ ]:
ENTITY = "seanv507"
PROJECT = "criteo_boost_sweep"
SWEEP_ID = "s15xvple"
TABLE_KEY = "sweep_column_statistics"

# 1. Read the 45 sweep runs.
api = wandb.Api()
sweep = api.sweep(f"{ENTITY}/{PROJECT}/{SWEEP_ID}")

rows = []
for run in sweep.runs:
    summary = dict(run.summary)
    rows.append({
        "learning_rate": run.config.get("learning_rate"),
        "max_depth": run.config.get("max_depth"),
        "subsample": run.config.get("subsample"),
        # This sweep stores the objective's minimum under this summary key.
        "validation_1-logloss": summary.get(
            "validation_1-logloss.min"
        ),
    })

# 2. Calculate one row per column.
values = pd.DataFrame(rows).apply(pd.to_numeric, errors="coerce")


In [ ]:
values.head()

In [ ]:


stats = (
    values.agg(["min", "max", "mean"])
    .T
    .reset_index()
    .rename(columns={"index": "column"})
)


In [ ]:
stats

In [ ]:

# 3. Log the derived table to a dedicated analysis run.
with wandb.init(
    entity=ENTITY,
    project=PROJECT,
    name=f"{SWEEP_ID}-column-statistics",
    job_type="analysis",
) as analysis_run:
    analysis_run.log({
        TABLE_KEY: wandb.Table(dataframe=stats)
    })
    analysis_run_id = analysis_run.id

# 4. Create a Report containing the table.
report = wr.Report(
    entity=ENTITY,
    project=PROJECT,
    title="Criteo boost sweep statistics",
    description=f"Column statistics across all runs in sweep {SWEEP_ID}.",
    blocks=[
        wr.H2("Sweep column statistics"),
        wr.PanelGrid(
            runsets=[
                wr.Runset(
                    entity=ENTITY,
                    project=PROJECT,
                    filters=[
                        wr.Metric("name").isin([analysis_run_id])
                    ],
                )
            ],
            panels=[
                wr.WeavePanelSummaryTable(
                    table_name=TABLE_KEY,
                    layout=wr.Layout(w=24, h=12),
                )
            ],
        ),
    ],
)

report.save()
print(report.url)



> **⚠️ Duplicate** — this cell redoes the entire pipeline from the `ENTITY = ...` cell above through the Report-building cell, self-contained in one cell. Same missing-`pandas`-import issue noted above applies here too.

In [ ]:

ENTITY = "seanv507"
PROJECT = "criteo_boost_sweep"
SWEEP_ID = "s15xvple"
TABLE_KEY = "sweep_column_statistics"

# 1. Read the 45 sweep runs.
api = wandb.Api()
sweep = api.sweep(f"{ENTITY}/{PROJECT}/{SWEEP_ID}")

rows = []
for run in sweep.runs:
    summary = dict(run.summary)
    rows.append({
        "learning_rate": run.config.get("learning_rate"),
        "max_depth": run.config.get("max_depth"),
        "subsample": run.config.get("subsample"),
        # This sweep stores the objective's minimum under this summary key.
        "validation_1-logloss": summary.get(
            "validation_1-logloss.min"
        ),
    })

# 2. Calculate one row per column.
values = pd.DataFrame(rows).apply(pd.to_numeric, errors="coerce")

stats = (
    values.agg(["min", "max", "mean"])
    .T
    .reset_index()
    .rename(columns={"index": "column"})
)

# 3. Log the derived table to a dedicated analysis run.
with wandb.init(
    entity=ENTITY,
    project=PROJECT,
    name=f"{SWEEP_ID}-column-statistics",
    job_type="analysis",
) as analysis_run:
    analysis_run.log({
        TABLE_KEY: wandb.Table(dataframe=stats)
    })
    analysis_run_id = analysis_run.id

# 4. Create a Report containing the table.
report = wr.Report(
    entity=ENTITY,
    project=PROJECT,
    title="Criteo boost sweep statistics",
    description=f"Column statistics across all runs in sweep {SWEEP_ID}.",
    blocks=[
        wr.H2("Sweep column statistics"),
        wr.PanelGrid(
            runsets=[
                wr.Runset(
                    entity=ENTITY,
                    project=PROJECT,
                    filters=[
                        wr.Metric("name").isin([analysis_run_id])
                    ],
                )
            ],
            panels=[
                wr.WeavePanelSummaryTable(
                    table_name=TABLE_KEY,
                    layout=wr.Layout(w=24, h=12),
                )
            ],
        ),
    ],
)

report.save()
print(report.url)
